# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MohinaRustamova/lyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

The rule: An article gets flagged for review if it's both stale (hasn't been touched in a while) and underperforming its position (its CTR is meaningfully below what similar-position articles typically get). Staleness alone isn't enough. A 14-month-old article still earning strong CTR doesn't need touching. Underperformance alone isn't enough either. A brand-new article naturally has thin data. The two signals together point at articles that are both neglected and showing it.

Reason codes the rule can output:

- STALE_AND_UNDERPERFORMING: both conditions true, top priority
- CTR_BELOW_POSITION_PEERS: CTR gap only, content itself may be fine, snippet likely the issue
- STALE_ONLY: old but still holding CTR, lower urgency
- NO_FLAG: neither condition met

Note on staleness signal: days_since_last_optimized turned out to have zero usable values at this decision date, every optimization in the warehouse happened after 2026-03-31, so there's nothing before the decision point to measure. Rather than drop the staleness idea, I substituted content_age_days (decision date minus content creation date) as the closest honest proxy for neglect, content that's been live a long time without necessarily being touched.

Signal 1: Staleness vs decline rate (ties to the refresh flag from the session)

In [9]:
# ---- Setup: load data and rebuild feature vector (matches ML-04 pattern) ----
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL, REPO_DIR = "https://github.com/MohinaRustamova/flyrank-ml-internship", "flyrank-ml-internship"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

%pip -q install duckdb

import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect()

if IN_COLAB:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
else:
    from getpass import getpass
    HF_TOKEN = getpass("HF_TOKEN: ")

con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

FACT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')"
DIMC = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"

DECISION_DATE = "2026-03-31"

# --- Current + prior 30-day windows, same dates as ML-04 ---
df = con.sql(f"""
    WITH current_win AS (
        SELECT content_hash_id,
               SUM(gsc_impressions) AS gsc_impressions,
               SUM(gsc_clicks) AS gsc_clicks,
               AVG(gsc_avg_position) AS gsc_avg_position
        FROM {FACT}
        WHERE report_date BETWEEN DATE '2026-03-02' AND DATE '2026-03-31'
          AND gsc_data_available IS TRUE
        GROUP BY content_hash_id
    ),
    prior_win AS (
        SELECT content_hash_id,
               SUM(gsc_impressions) AS impressions_prior30
        FROM {FACT}
        WHERE report_date BETWEEN DATE '2026-01-31' AND DATE '2026-03-01'
          AND gsc_data_available IS TRUE
        GROUP BY content_hash_id
    )
    SELECT c.content_hash_id,
           c.gsc_impressions, c.gsc_clicks, c.gsc_avg_position,
           p.impressions_prior30,
           CASE WHEN c.gsc_impressions < p.impressions_prior30 * 0.8
                THEN 1 ELSE 0 END AS is_declining_label
    FROM current_win c
    JOIN prior_win p USING (content_hash_id)
    WHERE p.impressions_prior30 > 0
""").df()

# --- Join dim_content for last_optimized_date ---
dim_content = con.sql(f"SELECT content_hash_id, last_optimized_date FROM {DIMC}").df()
df = df.merge(dim_content, on="content_hash_id", how="left")

# --- Leak-safe days_since_last_optimized (from ML-05) ---
decision_ts = pd.Timestamp(DECISION_DATE)
df["last_optimized_date"] = pd.to_datetime(df["last_optimized_date"])
known_optimization = df["last_optimized_date"].notna() & (df["last_optimized_date"] <= decision_ts)

df["days_since_last_optimized"] = np.where(
    known_optimization,
    (decision_ts - df["last_optimized_date"]).dt.days,
    np.nan
)

# --- content_age_days: honest staleness substitute (see Section 1 note) ---
dim_content2 = con.sql(f"SELECT content_hash_id, content_created_date FROM {DIMC}").df()
df = df.merge(dim_content2, on="content_hash_id", how="left")

df["content_created_date"] = pd.to_datetime(df["content_created_date"])
df["content_age_days"] = (decision_ts - df["content_created_date"]).dt.days

print(df.shape)
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(135113, 10)


,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,impressions_prior30,is_declining_label,last_optimized_date,days_since_last_optimized,content_created_date,content_age_days
0,content_e67934818ca184a1,1072.0,2.0,15.667524,320.0,0,2026-05-15,NaN,2025-04-14,351
1,content_9634c35544bcc47b,313.0,1.0,11.978349,411.0,1,2026-05-27,NaN,2025-04-14,351
2,content_4ece07fdea783709,653.0,0.0,17.465739,629.0,0,NaT,NaN,2025-04-14,351
3,content_4d9b25ca95147676,1028.0,18.0,4.869805,1421.0,1,NaT,NaN,2025-04-14,351
4,content_2dae25d4660d074a,1366.0,1.0,24.638455,1209.0,0,2026-05-27,NaN,2025-04-14,351


In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
bins = [0, 90, 180, 365, float("inf")]
labels = ["0-3mo", "3-6mo", "6-12mo", "12+mo"]

df["age_bucket"] = pd.cut(df["content_age_days"], bins=bins, labels=labels)

signal1 = (
    df.groupby("age_bucket", observed=True)
      .agg(n=("content_hash_id", "count"),
           share_declining=("is_declining_label", "mean"))
      .assign(share_declining=lambda x: (x["share_declining"] * 100).round(1))
)
print(signal1)

                n  share_declining
age_bucket                        
0-3mo       30660             20.2
3-6mo       23608             26.6
6-12mo      60894             25.7
12+mo       19951             21.8


Signal 2: CTR vs position peers (ties to the CTR fix / snippet flag)

In [11]:
MIN_IMPRESSIONS = 100  # eligibility floor: below this, CTR is too noisy to judge

eligible = df[df["gsc_impressions"] >= MIN_IMPRESSIONS].copy()
print("Eligible rows:", len(eligible), "of", len(df), f"({len(eligible)/len(df):.1%})")

pos_bins = [0, 3, 6, 10, 20, float("inf")]
pos_labels = ["1-3", "3-6", "6-10", "10-20", "20+"]

eligible["position_bucket"] = pd.cut(eligible["gsc_avg_position"], bins=pos_bins, labels=pos_labels)
eligible["ctr"] = eligible["gsc_clicks"] / eligible["gsc_impressions"]

peer_median = eligible.groupby("position_bucket", observed=True)["ctr"].transform("median")
eligible["ctr_below_peers"] = eligible["ctr"] < (peer_median * 0.5)

signal2 = (
    eligible.groupby("position_bucket", observed=True)
      .agg(n=("content_hash_id", "count"),
           median_ctr=("ctr", "median"),
           share_far_below=("ctr_below_peers", "mean"))
      .assign(median_ctr=lambda x: (x["median_ctr"] * 100).round(2),
              share_far_below=lambda x: (x["share_far_below"] * 100).round(1))
)
print(signal2)

Eligible rows: 86402 of 135113 (63.9%)
                     n  median_ctr  share_far_below
position_bucket                                    
1-3               6813        0.21             32.5
3-6              20182        0.21             31.9
6-10             19425        0.15             39.0
10-20            18887        0.09             45.2
20+              21094        0.00              0.0


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Updated rule note: Section 1 changed two things about the original rule. First, days_since_last_optimized was unusable at this decision date (see Signal 1 note), so "staleness" is measured by content_age_days instead. Second, since Signal 1 came back MIXED, age alone never triggers the top-priority reason code, it only acts as a secondary signal on top of a confirmed CTR gap. Third, since Signal 2 is mathematically undefined at position 20+, articles ranked below position 20 are excluded from CTR-based flags entirely and can only ever get NO_FLAG

In [12]:
MIN_IMPRESSIONS = 100          # eligibility floor, matches Signal 2 check
AGING_BUCKETS = ["3-6mo", "6-12mo"]  # buckets where decline rate was elevated in Signal 1

# --- Rebuild age_bucket (in case Signal 1 cell didn't run first) ---
age_bins = [0, 90, 180, 365, float("inf")]
age_labels = ["0-3mo", "3-6mo", "6-12mo", "12+mo"]
df["age_bucket"] = pd.cut(df["content_age_days"], bins=age_bins, labels=age_labels)

# --- Rebuild position_bucket and ctr (in case Signal 2 cell didn't run first) ---
pos_bins = [0, 3, 6, 10, 20, float("inf")]
pos_labels = ["1-3", "3-6", "6-10", "10-20", "20+"]
df["position_bucket"] = pd.cut(df["gsc_avg_position"], bins=pos_bins, labels=pos_labels)
df["ctr"] = df["gsc_clicks"] / df["gsc_impressions"]

# --- Eligibility (Population step, same idea as the deck) ---
df["eligible"] = (
    (df["gsc_impressions"] >= MIN_IMPRESSIONS) &
    (df["position_bucket"] != "20+")
)

# --- Peer median CTR per position bucket, computed only on eligible rows ---
peer_ctr_map = (
    df[df["eligible"]]
    .groupby("position_bucket", observed=True)["ctr"]
    .median()
)
df["peer_median_ctr"] = df["position_bucket"].map(peer_ctr_map)

# --- CTR flag: only meaningful where eligible ---
df["ctr_flag"] = df["eligible"] & (df["ctr"] < df["peer_median_ctr"] * 0.5)

# --- Aging flag: content in the buckets where decline rate was highest ---
df["aging_flag"] = df["age_bucket"].isin(AGING_BUCKETS)

# --- Reason code + action label ---
def assign_reason(row):
    if row["ctr_flag"] and row["aging_flag"]:
        return "STALE_AND_UNDERPERFORMING", "content_fix"
    elif row["ctr_flag"]:
        return "CTR_BELOW_POSITION_PEERS", "snippet_fix"
    elif row["aging_flag"]:
        return "STALE_ONLY", "monitor"
    else:
        return "NO_FLAG", "no_action"

df[["reason_code", "action"]] = df.apply(
    lambda r: pd.Series(assign_reason(r)), axis=1
)

# --- Score: expected clicks lost vs peers (stake-based, not just a flag) ---
expected_click_gap = (
    (df["peer_median_ctr"] - df["ctr"]).clip(lower=0) * df["gsc_impressions"]
).fillna(0)

df["score"] = (
    df["ctr_flag"].astype(int) * expected_click_gap * 2
    + df["aging_flag"].astype(int) * (df["gsc_impressions"] / 1000)
).round(1)

# --- Ranked queue ---
queue = df.sort_values("score", ascending=False).reset_index(drop=True)

print(queue["reason_code"].value_counts())
print()
print(queue[["content_hash_id", "score", "reason_code", "action",
             "gsc_impressions", "ctr", "peer_median_ctr", "position_bucket", "age_bucket"]].head(20))

import os
os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("\nSaved:", len(queue), "rows to work/outputs/baseline_action_score.csv")

reason_code
STALE_ONLY                   69717
NO_FLAG                      40634
STALE_AND_UNDERPERFORMING    14785
CTR_BELOW_POSITION_PEERS      9977
Name: count, dtype: int64

             content_hash_id  score                reason_code       action  \
0   content_34a70fea29d15f24  655.6  STALE_AND_UNDERPERFORMING  content_fix   
1   content_44f34c0a90047651  582.4   CTR_BELOW_POSITION_PEERS  snippet_fix   
2   content_8e1334d6356668e3  561.0   CTR_BELOW_POSITION_PEERS  snippet_fix   
3   content_7c6373141eae744a  510.1  STALE_AND_UNDERPERFORMING  content_fix   
4   content_f6116743b00afc2d  394.6  STALE_AND_UNDERPERFORMING  content_fix   
5   content_fec55986a1868d62  366.6   CTR_BELOW_POSITION_PEERS  snippet_fix   
6   content_fc67675904376267  275.8  STALE_AND_UNDERPERFORMING  content_fix   
7   content_1642f339bd6e7c8d  270.0  STALE_AND_UNDERPERFORMING  content_fix   
8   content_33d31496fca9665e  267.8  STALE_AND_UNDERPERFORMING  content_fix   
9   content_306bc78dff1eb683  2

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Top-20 review

- 34a70fea — content_fix, STALE_AND_UNDERPERFORMING. Confidence: high, 142K impressions with CTR at 14% of peer median is a large, real gap. Would be wrong if: the page recently changed URL/redirect and GSC hasn't fully re-attributed clicks yet.

- 44f34c0a — snippet_fix, CTR_BELOW_POSITION_PEERS. Confidence: high, 212K impressions, largest volume in the queue, CTR near-zero relative to peers. Would be wrong if: this is a rich-result/featured-snippet page where clicks route through a different UI element GSC undercounts.

- 8e1334d6 — snippet_fix, CTR_BELOW_POSITION_PEERS. Confidence: medium, CTR of 0.0007% at 134K impressions is almost implausibly low. Would be wrong if: this reflects a tracking/tagging break rather than a genuine content problem, worth a manual check before treating it as a snippet issue.
- 7c6373141 — content_fix, STALE_AND_UNDERPERFORMING. Confidence: high, 128K impressions, clear CTR gap, mid-age bucket (3-6mo) so aging is plausible, not a stretch.

- f6116743 — content_fix, STALE_AND_UNDERPERFORMING. Confidence: medium-high, solid volume and gap. Would be wrong if: position 6-10 for this query cluster is naturally low-CTR (e.g. heavy "People Also Ask" competition) rather than a content issue.

- fec55986 — snippet_fix, CTR_BELOW_POSITION_PEERS. Confidence: medium, same near-zero CTR pattern as row 3 (0.0008%). Would be wrong if: also a tracking artifact, worth checking alongside row 3 before batch-actioning.
- fc676759 — content_fix, STALE_AND_UNDERPERFORMING. Confidence: medium, smaller volume (59K) but real gap. Would be wrong if: this topic is seasonal and currently off-peak, the CTR dip could be temporary demand, not decay.
- 1642f339 — content_fix, STALE_AND_UNDERPERFORMING. Confidence: medium-high, consistent pattern with rows above.
- 33d31496 — content_fix, STALE_AND_UNDERPERFORMING. Confidence: medium, CTR gap is real but smaller relative to peer (42% of peer median vs. near-zero above), less severe than top rows.
- 306bc78d — snippet_fix, CTR_BELOW_POSITION_PEERS. Confidence: medium, 12+mo age plus CTR gap. Would be wrong if: this is evergreen content whose ranking query has simply gotten more competitive over time, a snippet rewrite wouldn't fix a competitive-landscape shift.
- a2a28625 — content_fix, STALE_AND_UNDERPERFORMING. Confidence: medium, smaller volume (64K), pattern consistent with row 4.
- 36fc1ee5 — content_fix, STALE_AND_UNDERPERFORMING. Confidence: medium, similar profile to row 5.
- 1bb7d17c — content_fix, STALE_AND_UNDERPERFORMING. Confidence: medium, lower volume than top rows, still a real gap.
- 895d440b — content_fix, STALE_AND_UNDERPERFORMING. Confidence: medium, CTR gap is the smallest of the STALE_AND_UNDERPERFORMING rows so far (44% of peer median), borderline case.
- e8a52cf3 — monitor, STALE_ONLY. Confidence: low, worth flagging as the weakest pick in the top 20, this row's CTR (0.27%) is actually above its position-bucket peer median (0.09%), it only made the queue because of high impressions × the aging weight, not because it's underperforming. Would be wrong to treat as urgent: the score here reflects volume, not a real problem, this is a good example of "what would make this wrong" being the entire point of the row.
- 37a6fac6 — content_fix, STALE_AND_UNDERPERFORMING. Confidence: medium-high, CTR at just 4% of peer median, one of the more severe gaps despite lower volume (47K).
- bb2a9972 — content_fix, STALE_AND_UNDERPERFORMING. Confidence: medium, consistent with the general pattern.
- 9b650e36 — content_fix, STALE_AND_UNDERPERFORMING. Confidence: medium, CTR gap smaller (36% of peer median), one of the weaker cases in the content_fix group.
- b2b85c28 — content_fix, STALE_AND_UNDERPERFORMING. Confidence: medium, similar to row 18, moderate rather than severe gap.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(queue[["content_hash_id", "score", "reason_code", "action",
             "gsc_impressions", "ctr", "peer_median_ctr", "position_bucket", "age_bucket"]].head(20).to_string())

             content_hash_id  score                reason_code       action  gsc_impressions       ctr  peer_median_ctr position_bucket age_bucket
0   content_34a70fea29d15f24  655.6  STALE_AND_UNDERPERFORMING  content_fix         142039.0  0.000289         0.002096             3-6     6-12mo
1   content_44f34c0a90047651  582.4   CTR_BELOW_POSITION_PEERS  snippet_fix         212115.0  0.000113         0.001486            6-10      0-3mo
2   content_8e1334d6356668e3  561.0   CTR_BELOW_POSITION_PEERS  snippet_fix         134264.0  0.000007         0.002096             3-6      12+mo
3   content_7c6373141eae744a  510.1  STALE_AND_UNDERPERFORMING  content_fix         128664.0  0.000614         0.002096             3-6      3-6mo
4   content_f6116743b00afc2d  394.6  STALE_AND_UNDERPERFORMING  content_fix         106408.0  0.000132         0.001486            6-10     6-12mo
5   content_fec55986a1868d62  366.6   CTR_BELOW_POSITION_PEERS  snippet_fix         124050.0  0.000008         0.00148

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks

The clearest weak pick is row 15, content_e8a52cf3d5988c07 (STALE_ONLY, monitor). Its CTR is 0.27%, which is actually above its position-bucket peer median of 0.09%, not below it. It only entered the top 20 because of high raw impressions (239K) combined with the aging weight in the score formula, not because it's genuinely underperforming. This is a case where the score rewarded volume alone. It's correctly labeled monitor rather than an active fix, but it shouldn't be read as "urgent," it's here because it's big, not because it's broken.

More broadly, every STALE_AND_UNDERPERFORMING and CTR_BELOW_POSITION_PEERS row in the top 20 shares one blind spot the rule can't see: seasonality. Rows 7 and 10 in particular sit in the 6-12mo and 12+mo age buckets, exactly where a real seasonal dip (the "home-office setup in July" example from the session) would look identical to genuine decay. The rule has no way to tell them apart. A human reviewer checking the actual topic and time of year before acting would catch this; the score alone cannot.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# --- Leakage check: confirm no current-window / future / product-flag columns fed the rule ---

RULE_INPUT_COLUMNS = [
    "gsc_impressions", "gsc_clicks", "gsc_avg_position",
    "content_age_days", "age_bucket", "position_bucket",
    "eligible", "ctr_flag", "aging_flag", "peer_median_ctr", "ctr",
]

print("Columns fed into score/reason_code/action:")
print(RULE_INPUT_COLUMNS)
print()

# Check 1: impressions_current30 / anything from the label window itself never entered the rule
leaky_terms = ["current30", "is_declining_label"]
found = [c for c in RULE_INPUT_COLUMNS if any(t in c for t in leaky_terms)]
print("Label-window columns found in rule inputs (should be empty):", found)

# Check 2: no FlyRank product flags (provider_used, model_used, or anything product-generated) used
product_flag_terms = ["provider_used", "model_used", "flag_snippet_review", "reason_code"]
found2 = [c for c in RULE_INPUT_COLUMNS if c in product_flag_terms]
print("Product-flag columns found in rule inputs (should be empty):", found2)

# Check 3: confirm days_since_last_optimized (which had a leak-safety issue earlier) is NOT used in the final score
print("days_since_last_optimized in rule inputs (should be False):",
      "days_since_last_optimized" in RULE_INPUT_COLUMNS)

# Check 4: gsc_impressions used in the score is the CURRENT window only, prior30 is not fed into scoring directly
print("impressions_prior30 in rule inputs (should be False):",
      "impressions_prior30" in RULE_INPUT_COLUMNS)

Columns fed into score/reason_code/action:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'content_age_days', 'age_bucket', 'position_bucket', 'eligible', 'ctr_flag', 'aging_flag', 'peer_median_ctr', 'ctr']

Label-window columns found in rule inputs (should be empty): []
Product-flag columns found in rule inputs (should be empty): []
days_since_last_optimized in rule inputs (should be False): False
impressions_prior30 in rule inputs (should be False): False


Leakage check result: The rule only uses current-window GSC metrics (gsc_impressions, gsc_clicks, gsc_avg_position), content_age_days (a static fact as of the decision date), and values derived purely from those (buckets, flags, peer medians). No column from the label window (impressions_current30 beyond what defines the current snapshot itself, or is_declining_label) feeds the score. days_since_last_optimized was investigated in Section 1 but excluded from the final rule since it had zero usable values at this decision date; only its honest substitute (content_age_days) was used. No FlyRank product flags were used as inputs, matching the exclusion documented back in ML-04.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.